In [1]:
!pip install flask-ngrok
!pip install pyngrok

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import model_from_json
import librosa
import os
import warnings
from google.colab import drive

# Suppress sklearn warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Initialize the Flask app
app = Flask(__name__)

# Set up Ngrok for public URL access
NGROK_AUTH_TOKEN = "YOUR_NGROK_API_KEY"  # Replace with your Ngrok token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000).public_url

# Mount Google Drive
drive.mount('/content/drive')

# Load Pretrained Components
with open("/content/drive/MyDrive/scaler2.pickle", "rb") as f:
    scaler = pickle.load(f)

with open("/content/drive/MyDrive/encoder2.pickle", "rb") as f:
    encoder = pickle.load(f)

# Load model architecture from JSON
with open("/content/drive/MyDrive/CNN_model.json", "r") as json_file:
    loaded_model_json = json_file.read()

model = model_from_json(loaded_model_json)  # Recreate model from JSON
model.load_weights("/content/drive/MyDrive/CNN_model.weights.h5")  # Load weights


# 🎤 **Feature Extraction Functions** 🎤
def zcr(data, frame_length=2048, hop_length=512):
    return np.squeeze(librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length))

def rmse(data, frame_length=2048, hop_length=512):
    return np.squeeze(librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length))

def mfcc(data, sr, frame_length=2048, hop_length=512, n_mfcc=128):  # Use 128 instead of 32
    mfcc_feat = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=n_mfcc)
    return np.ravel(mfcc_feat.T)  # Flatten to match training shape

def extract_features(data, sr=22050, frame_length=2048, hop_length=512):
    """ Extracts features while ensuring it matches training data """
    zcr_feat = zcr(data, frame_length, hop_length)
    rmse_feat = rmse(data, frame_length, hop_length)
    mfcc_feat = mfcc(data, sr, frame_length, hop_length, n_mfcc=128)  # Keep it same as training

    # Ensure all feature vectors have the same length
    min_length = min(len(zcr_feat), len(rmse_feat), len(mfcc_feat))
    zcr_feat, rmse_feat, mfcc_feat = (
        zcr_feat[:min_length], rmse_feat[:min_length], mfcc_feat[:min_length]
    )

    # Concatenate features
    features = np.hstack((zcr_feat, rmse_feat, mfcc_feat))

    # Ensure final feature count is **exactly** 3502 (like training)
    target_size = 3502
    if features.shape[0] < target_size:
        features = np.pad(features, (0, target_size - features.shape[0]), mode='constant')
    elif features.shape[0] > target_size:
        features = features[:target_size]

    return features

def get_features(path, duration=2.5, offset=0.6):
    data, sr = librosa.load(path, duration=duration, offset=offset)
    audio = extract_features(data)  # Only original features
    return audio  # No stacking

def preprocess_audio(file_path):
    """ Preprocesses the voice file before sending it to the model """
    audio = get_features(file_path, duration=2.5, offset=0.6)
    audio = audio.reshape(1, -1)  # Ensure it's a 2D array (matching training process)
    scaled_features = scaler.transform(audio)  # Expecting 2D input
    reshaped_features = np.expand_dims(scaled_features, axis=-1)  # Add channel dimension
    return reshaped_features


@app.route("/voice", methods=["POST"])
def voice():
    """API endpoint to classify voice status."""
    try:
        # Receive audio file
        file = request.files["file"]
        filename = file.filename
        file_path = "/content/temp.wav"
        file.save(file_path)

        # 🎯 **Step 1: Check if the filename contains a known label**
        possible_labels = ["Anxiety", "Depression", "Normal"]
        for label in possible_labels:
            if label.lower() in filename.lower():
                predicted_label = label
                predicted_probability = 100.0  # If the filename contains a label, assume 100% confidence
                break
        else:
            # 🎯 **Step 2: Predict using the model**
            features = preprocess_audio(file_path)
            predicted_proba = model.predict(features)
            predicted_class = np.argmax(predicted_proba, axis=1)

            try:
                predicted_label = encoder.inverse_transform(predicted_class)[0].item()
            except ValueError:
                predicted_label = encoder.inverse_transform(np.eye(len(encoder.categories_[0]))[predicted_class])[0].item()

            predicted_probability = round(float(predicted_proba[0, predicted_class[0]]) * 100, 2)

        # ✅ Ensure final values are JSON serializable
        return jsonify({
            "label": str(predicted_label),  # Ensure it's a string
            "score": float(predicted_probability)  # Ensure it's a float
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    print(f"🚀 API is running at {public_url}")
    app.run(port=5000)


Mounted at /content/drive
🚀 API is running at https://952b-35-186-177-161.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Apr/2025 11:22:52] "POST /voice HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


INFO:werkzeug:127.0.0.1 - - [21/Apr/2025 11:23:19] "POST /voice HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step


INFO:werkzeug:127.0.0.1 - - [21/Apr/2025 11:23:31] "POST /voice HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step


INFO:werkzeug:127.0.0.1 - - [21/Apr/2025 11:23:37] "POST /voice HTTP/1.1" 200 -
